# SmartMoney – Fase 6A: clasificador de INTENCIÓN del asistente

**Objetivo:** dada una frase que el usuario le escribe al asistente ("cuanto gaste en comida",
"me alcanza para un cdt", "que es el interes compuesto"), predecir su **intención** entre 6:

`consultar_datos`, `simular_escenario`, `aprender_concepto`, `pedir_consejo`, `ayuda_app`, `conversacion`.

**Diseño experimental (idéntico en espíritu a la Fase 3)**
- **Entrenamiento:** `dataset_intenciones_entrenamiento.csv` (frases escritas por el equipo, ver
  `guia_etiquetado.md`). **No** contiene preguntas del formulario.
- **Selección de hiperparámetros:** validación cruzada de 5 particiones **solo con el entrenamiento**.
- **Prueba:** las preguntas **reales** del formulario, etiquetadas a mano por dos personas
  (`para_etiquetar_A.csv` y `para_etiquetar_B.csv`), con desacuerdos resueltos en
  `resolucion_desacuerdos.csv`. Se usan **una sola vez**, al final.
- **Modelos comparados:**
  1. Reglas de palabras clave (línea base sin aprendizaje).
  2. Naive Bayes multinomial sobre TF-IDF `char_wb`.
  3. **Regresión logística sobre TF-IDF de n-gramas de caracteres** (modelo propuesto).
  4. Regresión logística con **caracteres + palabras** (variante).
- **Métricas:** accuracy, F1 macro con IC 95 % por bootstrap, F1 por intención, matriz de
  confusión, prueba de McNemar contra la línea base y cobertura/acierto por umbral de confianza.
- **Salida:** `modelo_intencion.json` con **el mismo formato** que `modelo_clasificador.json`
  (Fase 3), para poder reutilizar `modelo.ts` sin cambios en TypeScript.

## 1. Archivos

Sube `dataset_intenciones_entrenamiento.csv`, `para_etiquetar_A.csv` y `para_etiquetar_B.csv`
cuando lo pida la celda (o ponlos en la misma carpeta si corres local).
`resolucion_desacuerdos.csv` no es necesario subirlo: el notebook lo crea si hay desacuerdos.

## 2. Datos de entrenamiento

## 3. Validación cruzada (solo con el entrenamiento)

5 particiones estratificadas. **Advertencia:** la validación sale alta porque las frases de
entrenamiento comparten estilo y vocabulario; el resultado que vale es el de las preguntas reales.

- **TF-IDF de n-gramas de caracteres (`char_wb`):** parte cada palabra en pedacitos de 2 a 5
  letras, así "cuanto"/"cuantto" y "ahorro"/"ahoro" comparten pedacitos (tolera typos).
- **Naive Bayes:** línea base de aprendizaje, rápida.
- **Regresión logística:** modelo propuesto (`C` controla qué tanto se ajusta).
- **Caracteres + palabras:** variante que añade n-gramas de palabras (1–2).

## 4. Línea base: reglas de palabras clave

Es lo que haría la app sin IA: busca palabras clave del mensaje, cuenta coincidencias y elige
la intención con más aciertos; si no encuentra ninguna, asigna `conversacion`. En empate se usa
el orden de desempate de `guia_etiquetado.md`. Las palabras salen de esa misma guía.

## 5. Exportar el modelo elegido por validación cruzada

El modelo que se exporta se elige por **F1 macro en validación cruzada**, no con las preguntas
reales (igual que en la Fase 3). Así, cuando llegue la evaluación con el formulario, esas
preguntas se usan **una sola vez**.

Se exporta el mejor modelo **lineal sobre TF-IDF de caracteres**, porque es el único que se
puede replicar con el `modelo.ts` de la Fase 4 (char_wb + TF-IDF + softmax). El formato del
JSON es **idéntico** al de `modelo_clasificador.json`; lo único que cambia es que `clases` son
las 6 intenciones (con id numérico 1..6 y su nombre en `nombres`).

## 6. Etiquetas reales: acuerdo entre etiquetadores (kappa de Cohen)

Aquí es donde el notebook **se detiene** para la revisión manual. Las preguntas del formulario
las etiquetan dos personas **por separado** (`para_etiquetar_A.csv` y `para_etiquetar_B.csv`).
El notebook:

1. Calcula el **kappa de Cohen** global y la matriz de desacuerdos.
2. Construye la etiqueta final: donde coinciden, esa; donde no, la de `resolucion_desacuerdos.csv`.
3. Si `resolucion_desacuerdos.csv` no existe, **lo crea** con los ids en desacuerdo y se detiene
   con un mensaje claro para que los etiquetadores lo llenen juntos.

> Esta parte no toca el entrenamiento ni la selección del modelo: es la prueba reservada.

## 7. Evaluación con las preguntas reales (una sola vez)

Se calculan todas las métricas de una sola pasada: accuracy, F1 macro con IC 95 % por bootstrap,
F1 por intención, matriz de confusión, McNemar contra la línea base y cobertura por umbral.

## 8. Escenario "frases nuevas"

Solo las preguntas cuyo texto normalizado **no** aparece en el entrenamiento. Es el escenario más
honesto: mide qué tan bien generaliza el modelo a frases que nunca vio.

## 9. Exportar resultados

Se guardan las tablas de resultados en CSV (y se descargan si corres en Colab).
`modelo_intencion.json` ya quedó guardado en la sección 5.

## 10. Conclusiones (calculadas automáticamente)

Nada de números escritos a mano: todo sale de las variables del notebook.

In [ ]:
import os
ARCHIVOS = ["dataset_intenciones_entrenamiento.csv", "para_etiquetar_A.csv", "para_etiquetar_B.csv"]
faltan = [a for a in ARCHIVOS if not os.path.exists(a)]
if faltan:
    try:
        from google.colab import files
        print("Sube:", faltan)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Faltan {faltan} en la carpeta actual")
print("Archivos listos:", [a for a in ARCHIVOS if os.path.exists(a)])

In [ ]:
import os, re, json, math, random, unicodedata
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, cohen_kappa_score)
from scipy.stats import binomtest

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)

# Las 6 intenciones. En el JSON se guardan con id numérico para reutilizar modelo.ts tal cual.
INTENCIONES = ["consultar_datos", "simular_escenario", "aprender_concepto",
               "pedir_consejo", "ayuda_app", "conversacion"]
CLASE = {n: i + 1 for i, n in enumerate(INTENCIONES)}     # nombre -> id numérico
NOMBRE = {i + 1: n for i, n in enumerate(INTENCIONES)}    # id numérico -> nombre
ETIQ = [CLASE[n] for n in INTENCIONES]
CORTO = {CLASE[n]: n.replace("_", " ") for n in INTENCIONES}


def normalizar(s):
    """Minúsculas, sin tildes, solo letras y números. Igual que Fase 3 y modelo.ts."""
    s = unicodedata.normalize("NFD", str(s).lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

In [ ]:
ent = pd.read_csv("dataset_intenciones_entrenamiento.csv")
ent["t"] = ent.texto.map(normalizar)
ent["clase"] = ent.intencion.map(CLASE)
assert ent.clase.notna().all(), "Hay intenciones que no coinciden con las 6 definidas."
assert ent.t.str.len().gt(0).all(), "Hay textos vacíos tras normalizar."
print("Entrenamiento:", len(ent), "frases")
print(ent.intencion.value_counts().to_string())
X, y = ent.texto.values, ent.clase.values

In [ ]:
def vec_char(**kw):
    return TfidfVectorizer(preprocessor=normalizar, analyzer="char_wb",
                           sublinear_tf=True, min_df=2, **kw)

def vec_palabra(**kw):
    return TfidfVectorizer(preprocessor=normalizar, analyzer="word",
                           sublinear_tf=True, min_df=2, **kw)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

busqueda_lr = GridSearchCV(
    Pipeline([("tfidf", vec_char()), ("clf", LogisticRegression(max_iter=3000))]),
    {"tfidf__ngram_range": [(2, 4), (2, 5), (3, 5)], "clf__C": [1, 5, 10, 30]},
    scoring="f1_macro", cv=cv, n_jobs=-1).fit(X, y)

busqueda_mix = GridSearchCV(
    Pipeline([("tfidf", FeatureUnion([("c", vec_char(ngram_range=(2, 5))),
                                      ("w", vec_palabra(ngram_range=(1, 2)))])),
              ("clf", LogisticRegression(max_iter=3000))]),
    {"clf__C": [1, 5, 10, 30]}, scoring="f1_macro", cv=cv, n_jobs=-1).fit(X, y)

busqueda_nb = GridSearchCV(
    Pipeline([("tfidf", vec_char()), ("clf", MultinomialNB())]),
    {"tfidf__ngram_range": [(2, 4), (2, 5), (3, 5)], "clf__alpha": [0.01, 0.05, 0.1, 0.5]},
    scoring="f1_macro", cv=cv, n_jobs=-1).fit(X, y)

print("RL (char)        :", busqueda_lr.best_params_, f"F1 macro CV = {busqueda_lr.best_score_:.3f}")
print("RL (char+palabra):", busqueda_mix.best_params_, f"F1 macro CV = {busqueda_mix.best_score_:.3f}")
print("Naive Bayes      :", busqueda_nb.best_params_, f"F1 macro CV = {busqueda_nb.best_score_:.3f}")
print("\nOjo: la validación es con datos del mismo estilo que el entrenamiento, así que sale alta. "
      "El resultado que vale es el de las preguntas reales.")

modelo_lr = busqueda_lr.best_estimator_     # ya reentrenado con todo el entrenamiento
modelo_nb = busqueda_nb.best_estimator_
modelo_mix = busqueda_mix.best_estimator_

In [ ]:
REGLAS = {
 "simular_escenario": ["me alcanza", "alcanza para", "si ahorro", "en cuanto tiempo",
     "cuanto tiempo", "puedo pagar", "cuota de", "si dejo de", "si invierto", "cuanto tendria",
     "si guardo", "si reduzco", "cuanto necesito", "si empiezo", "me rinde", "cuanto me sobra",
     "si gano", "calculame", "cuanto tengo que ahorrar", "cuanto deberia ahorrar",
     "cuanto debo ahorrar", "si aporto", "cuanto podria ahorrar", "si bajo", "me sobra si"],
 "pedir_consejo": ["me recomiendas", "consejo", "aconsejas", "me conviene", "esta bien",
     "vale la pena", "deberia", "como hago para", "que hago", "como evito", "estrategia",
     "estoy gastando", "que puedo hacer", "como controlo", "como organizo", "es malo",
     "es buena idea", "que me aconsejas", "dame un consejo"],
 "consultar_datos": ["cuanto gaste", "cuanto he gastado", "cuanto llevo", "total de",
     "muestrame", "lista mis", "resumen", "como voy con", "cuanto me falta", "cual fue mi",
     "cual es mi", "que dia gaste", "en que he gastado", "cuanto ingrese", "cuanto he ahorrado",
     "cuantos movimientos", "balance", "mis gastos", "mis ingresos", "promedio que gasto",
     "cuanto ahorre", "que categoria", "cuanto tengo disponible", "que metas tengo"],
 "aprender_concepto": ["que es", "como funciona", "explicame", "que significa",
     "en que consiste", "para que sirve", "diferencia entre", "que es mejor", "como se calcula",
     "de que depende", "quiero entender", "cuentame sobre", "hablame de", "quiero aprender",
     "ayudame a entender"],
 "ayuda_app": ["la app", "aplicacion", "registrar un gasto", "registrar un ingreso",
     "crear una meta", "editar una meta", "contrasena", "exportar", "recordatorio",
     "notificaciones", "cerrar sesion", "mi cuenta", "segura", "funciones", "modo oscuro",
     "sincronizar", "descargar mis", "categorizar", "limite de gasto", "historial",
     "como registro", "donde veo", "como cambio", "como activo", "como apago", "como uso",
     "como conecto", "mis datos estan", "que puedes hacer"],
 "conversacion": ["hola", "buenas", "gracias", "chao", "adios", "no entendi", "quien eres",
     "eres un", "que tal", "bye", "listo", "de una", "jaja", "como estas", "que hora es",
     "no capte", "explicamelo"],
}
PRIORIDAD = ["simular_escenario", "pedir_consejo", "consultar_datos",
             "aprender_concepto", "ayuda_app", "conversacion"]


def predecir_reglas(textos):
    pred = []
    for t in textos:
        tt = " " + normalizar(t) + " "
        votos = {k: sum(1 for kw in kws if (" " + normalizar(kw) + " ") in tt)
                 for k, kws in REGLAS.items()}
        m = max(votos.values())
        pred.append(CLASE["conversacion"] if m == 0 else CLASE[next(k for k in PRIORIDAD if votos[k] == m)])
    return np.array(pred)


ejemplos = ["cuanto gaste en comida", "me alcanza para un cdt", "que es el interes compuesto",
            "me recomiendas para ahorrar", "como cambio mi contrasena", "hola gracias"]
print("Ejemplos de la línea base:")
for t, p in zip(ejemplos, predecir_reglas(ejemplos)):
    print(f"  {t:34s} -> {NOMBRE[p]}")

In [ ]:
tfidf, clf = modelo_lr.named_steps["tfidf"], modelo_lr.named_steps["clf"]
vocab = tfidf.vocabulary_
orden = sorted(vocab, key=vocab.get)                      # n-gramas en el orden de las columnas

modelo_json = {
    "version": "clasificador-intenciones-v1",
    "descripcion": "TF-IDF char_wb + regresión logística multinomial. 6 intenciones del asistente.",
    "preprocesamiento": "minúsculas → NFD sin marcas diacríticas → [^a-z0-9]+ a espacio → trim",
    "ngram_range": list(tfidf.ngram_range),
    "analizador": "char_wb: por cada palabra w se usa ' '+w+' '; para n de nmin a nmax: si len <= n se agrega la palabra completa una vez y se termina; si no, todos sus n-gramas de tamaño n",
    "sublinear_tf": True, "norma": "l2",
    "clases": [int(c) for c in clf.classes_],
    "nombres": {int(c): NOMBRE[int(c)] for c in clf.classes_},
    "ngramas": orden,
    "idf": [round(float(v), 6) for v in tfidf.idf_],
    "coef": [[round(float(v), 6) for v in fila] for fila in clf.coef_],
    "intercepto": [round(float(v), 6) for v in clf.intercept_],
}


def ngramas_char_wb(t, nmin, nmax):
    out = []
    for w in t.split():
        w = " " + w + " "
        for n in range(nmin, nmax + 1):
            if len(w) <= n:
                out.append(w); break
            out.extend(w[i:i + n] for i in range(len(w) - n + 1))
    return out


def predecir_manual(texto, m):
    """Predictor a mano, sin scikit-learn (referencia exacta para TypeScript)."""
    idx = {g: i for i, g in enumerate(m["ngramas"])}
    cuenta = {}
    for g in ngramas_char_wb(normalizar(texto), *m["ngram_range"]):
        if g in idx:
            cuenta[idx[g]] = cuenta.get(idx[g], 0) + 1
    x = {i: (1 + math.log(c)) * m["idf"][i] for i, c in cuenta.items()}
    nrm = math.sqrt(sum(v * v for v in x.values())) or 1.0
    x = {i: v / nrm for i, v in x.items()}
    z = [m["intercepto"][k] + sum(m["coef"][k][i] * v for i, v in x.items())
         for k in range(len(m["clases"]))]
    mx = max(z); e = [math.exp(v - mx) for v in z]; s = sum(e)
    return {m["clases"][k]: e[k] / s for k in range(len(e))}


# Verificación: el predictor a mano debe dar EXACTAMENTE lo mismo que scikit-learn
muestra = ent.texto.tolist()
p_sk = modelo_lr.predict_proba(muestra)
p_man = np.array([[predecir_manual(t, modelo_json)[c] for c in modelo_json["clases"]] for t in muestra])
dif = np.abs(p_sk - p_man).max()
iguales = (p_sk.argmax(1) == p_man.argmax(1)).mean()
print(f"Diferencia máxima de probabilidad: {dif:.2e} | misma predicción en {iguales:.1%} de {len(muestra)} frases")
assert iguales == 1.0 and dif < 1e-3

# Casos de prueba (uno por intención) para verificar la réplica en TypeScript
casos = ["cuanto gaste en comida este mes", "me alcanza para un cdt si tengo 500 mil",
         "que es el interes compuesto", "me conviene ahorrar o invertir",
         "como cambio mi contrasena", "hola parce gracias"]
modelo_json["casos_prueba"] = []
for t in casos:
    pr = predecir_manual(t, modelo_json); c = max(pr, key=pr.get)
    modelo_json["casos_prueba"].append({"texto": t, "normalizado": normalizar(t),
                                        "grupo_id": c, "probabilidad": round(pr[c], 4)})
    print(f"{t:40s} -> {NOMBRE[c]:18s} {pr[c]:.2f}")

with open("modelo_intencion.json", "w", encoding="utf-8") as f:
    json.dump(modelo_json, f, ensure_ascii=False, separators=(",", ":"))
print(f"\nmodelo_intencion.json: {len(orden)} n-gramas x {len(modelo_json['clases'])} clases, "
      f"{os.path.getsize('modelo_intencion.json') / 1024:.0f} KB")

In [ ]:
A = pd.read_csv("para_etiquetar_A.csv")
B = pd.read_csv("para_etiquetar_B.csv")


def sin_etiquetar(df):
    return int(df["intencion"].fillna("").astype(str).str.strip().eq("").sum())


if sin_etiquetar(A) or sin_etiquetar(B):
    raise RuntimeError(
        "PENDIENTE DE ETIQUETADO: falta llenar la columna 'intencion' de "
        f"para_etiquetar_A.csv ({sin_etiquetar(A)} sin llenar) y "
        f"para_etiquetar_B.csv ({sin_etiquetar(B)} sin llenar).\n"
        "Usa SOLO los 6 valores de guia_etiquetado.md y vuelve a ejecutar el notebook.\n"
        "El notebook se detiene aquí a propósito: las preguntas reales se usan UNA sola vez.")

M = A[["id", "texto", "intencion"]].merge(B[["id", "intencion"]], on="id", suffixes=("_A", "_B"))
assert len(M) == len(A) == len(B), "Los archivos A y B deben traer exactamente los mismos ids."

kappa = cohen_kappa_score(M.intencion_A, M.intencion_B)
print(f"Acuerdo entre etiquetadores: kappa de Cohen = {kappa:.3f}")
print("\nMatriz de desacuerdos (filas = A, columnas = B):")
print(pd.crosstab(M.intencion_A, M.intencion_B).to_string())

desacuerdos = M[M.intencion_A != M.intencion_B]
print(f"\nFrases en desacuerdo: {len(desacuerdos)} de {len(M)} ({len(desacuerdos) / len(M):.1%})")

RUTA_RES = "resolucion_desacuerdos.csv"
if not os.path.exists(RUTA_RES):
    desacuerdos[["id"]].assign(intencion="").to_csv(RUTA_RES, index=False, encoding="utf-8-sig")
    raise RuntimeError(
        f"Se creó {RUTA_RES} con los {len(desacuerdos)} ids en desacuerdo. "
        "Llénalo junto con el otro etiquetador (columna 'intencion') y vuelve a ejecutar el notebook.")

res = pd.read_csv(RUTA_RES)
faltan_res = int(res["intencion"].fillna("").astype(str).str.strip().eq("").sum())
if faltan_res:
    raise RuntimeError(f"Faltan {faltan_res} etiquetas en {RUTA_RES}. Complétalo y vuelve a ejecutar.")
resol = dict(zip(res.id, res.intencion))

M["intencion_final"] = [a if a == b else resol.get(i, "") for i, a, b in zip(M.id, M.intencion_A, M.intencion_B)]
assert M.intencion_final.isin(INTENCIONES).all(), "Hay etiquetas finales fuera de las 6 intenciones."

M["clase"] = M.intencion_final.map(CLASE)
M["t"] = M.texto.map(normalizar)
print(f"Etiqueta final lista: {len(M)} preguntas reales")

In [ ]:
def bootstrap_f1(y, p, n=2000, semilla=SEMILLA):
    r = np.random.default_rng(semilla); y, p = np.asarray(y), np.asarray(p); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        v.append(f1_score(y[i], p[i], average="macro", labels=ETIQ, zero_division=0))
    return np.percentile(v, [2.5, 97.5])


MODELOS = {
    "Reglas (línea base)": lambda d: predecir_reglas(d.texto.values),
    "Naive Bayes (char)": lambda d: modelo_nb.predict(d.texto),
    "Regresión logística (char)": lambda d: modelo_lr.predict(d.texto),
    "Regresión logística (char+palabra)": lambda d: modelo_mix.predict(d.texto),
}

d = M.copy()                       # todas las preguntas reales
IC, PRED = {}, {}
filas = []
for nom, f in MODELOS.items():
    p = f(d); PRED[nom] = p
    lo, hi = bootstrap_f1(d.clase, p); IC[nom] = (lo, hi)
    filas.append({"Modelo": nom, "n": len(d),
                  "Accuracy": accuracy_score(d.clase, p),
                  "F1 macro": f1_score(d.clase, p, average="macro", labels=ETIQ, zero_division=0),
                  "IC 95 %": f"[{lo:.3f}, {hi:.3f}]"})
resultados = pd.DataFrame(filas)
resultados.style.format({"Accuracy": "{:.3f}", "F1 macro": "{:.3f}"}).hide(axis="index")

In [ ]:
# Prueba de McNemar: ¿la regresión logística le gana a la línea base de reglas?
def mcnemar(y, a, b):
    y, a, b = map(np.asarray, (y, a, b))
    a_ok, b_ok = a == y, b == y
    b01 = int(np.sum(a_ok & ~b_ok))     # el modelo acierta y la base falla
    b10 = int(np.sum(~a_ok & b_ok))     # la base acierta y el modelo falla
    n = b01 + b10
    return b01, b10, (1.0 if n == 0 else binomtest(b01, n, 0.5).pvalue)


b01, b10, pval = mcnemar(d.clase, PRED["Regresión logística (char)"], PRED["Reglas (línea base)"])
print(f"McNemar  RL(char) vs Reglas")
print(f"  modelo acierta y reglas falla: {b01}")
print(f"  reglas acierta y modelo falla: {b10}")
print(f"  p-valor exacto (binomial, 2 colas): {pval:.4f}  ->  "
      f"{'diferencia significativa al 5 %' if pval < 0.05 else 'NO concluyente'}")

# F1 por intención
tabla = pd.DataFrame({"Intencion": [NOMBRE[g] for g in ETIQ],
                      "n": [int((d.clase == g).sum()) for g in ETIQ]})
for nom in MODELOS:
    tabla[nom] = f1_score(d.clase, PRED[nom], average=None, labels=ETIQ, zero_division=0)
tabla["Nota"] = np.where(tabla.n < 15, "indicativo (n < 15)", "")
tabla.style.format({k: "{:.2f}" for k in MODELOS}).hide(axis="index")

In [ ]:
# Matriz de confusión (regresión logística char), proporción por fila
cm = confusion_matrix(d.clase, PRED["Regresión logística (char)"], labels=ETIQ, normalize="true")
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=[CORTO[g] for g in ETIQ]).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_xlabel("Intención predicha"); ax.set_ylabel("Intención real (formulario)")
ax.set_title("Regresión logística (char) – proporción por fila")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig("matriz_confusion_intencion.png", dpi=200); plt.show()

In [ ]:
# Cobertura y acierto según el umbral de confianza máxima
proba = modelo_lr.predict_proba(d.texto.values); pmax = proba.max(1)
acierto = modelo_lr.classes_[proba.argmax(1)] == d.clase.values
print("Umbral de confianza -> cobertura (cuántas preguntas cubre) y acierto:")
for u in [0.5, 0.6, 0.7, 0.8]:
    m = pmax >= u
    acc = acierto[m].mean() if m.any() else float("nan")
    print(f"  confianza >= {u:.1f}: cubre {m.mean():5.1%} de las preguntas, acierta {acc:5.1%}")

In [ ]:
vistos = set(ent.t)
nuevas = M[~M.t.isin(vistos)].copy()
print(f"Preguntas cuyo texto no aparece en el entrenamiento: {len(nuevas)} de {len(M)} ({len(nuevas) / len(M):.1%})\n")
filas_nuevas = []
for nom, f in MODELOS.items():
    p = f(nuevas); lo, hi = bootstrap_f1(nuevas.clase, p)
    filas_nuevas.append({"Modelo": nom, "n": len(nuevas),
                         "Accuracy": accuracy_score(nuevas.clase, p),
                         "F1 macro": f1_score(nuevas.clase, p, average="macro", labels=ETIQ, zero_division=0),
                         "IC 95 %": f"[{lo:.3f}, {hi:.3f}]"})
resultados_nuevas = pd.DataFrame(filas_nuevas)
resultados_nuevas.style.format({"Accuracy": "{:.3f}", "F1 macro": "{:.3f}"}).hide(axis="index")

In [ ]:
resultados.to_csv("resultados_fase6.csv", index=False, encoding="utf-8-sig")
tabla.to_csv("f1_por_intencion.csv", index=False, encoding="utf-8-sig")
print("resultados_fase6.csv (todas las preguntas):")
print(resultados.round(3).to_string(index=False))
print("\nf1_por_intencion.csv:")
print(tabla.to_string(index=False))
try:
    from google.colab import files
    for a in ["modelo_intencion.json", "resultados_fase6.csv", "f1_por_intencion.csv",
              "matriz_confusion_intencion.png"]:
        files.download(a)
except ImportError:
    pass

In [ ]:
acc_lr = accuracy_score(d.clase, PRED["Regresión logística (char)"])
f1_lr = f1_score(d.clase, PRED["Regresión logística (char)"], average="macro", labels=ETIQ, zero_division=0)
f1_rg = f1_score(d.clase, PRED["Reglas (línea base)"], average="macro", labels=ETIQ, zero_division=0)
lo_lr, hi_lr = IC["Regresión logística (char)"]
peor = tabla.sort_values("Regresión logística (char)").iloc[0]
mejor = tabla.sort_values("Regresión logística (char)", ascending=False).iloc[0]
acc_nuevas = accuracy_score(nuevas.clase, PRED["Regresión logística (char)"]) if len(nuevas) else float("nan")

print(f"1. Acuerdo entre etiquetadores: kappa = {kappa:.3f} "
      f"({'aceptable (>= 0.6)' if kappa >= 0.6 else 'bajo: conviene revisar la guía y re-etiquetar'}).")
print(f"2. Datos de prueba: {len(M)} preguntas reales del formulario, etiquetadas a mano por dos "
      f"personas; {len(nuevas)} de ellas ({len(nuevas) / len(M):.1%}) no estaban en el entrenamiento.")
print(f"3. Regresión logística (char) sobre las preguntas reales: accuracy = {acc_lr:.3f}, "
      f"F1 macro = {f1_lr:.3f} (IC 95 % [{lo_lr:.3f}, {hi_lr:.3f}]).")
print(f"4. Línea base de reglas: F1 macro = {f1_rg:.3f}; McNemar p = {pval:.4f} "
      f"({'la diferencia es significativa' if pval < 0.05 else 'la diferencia NO es concluyente'}).")
print(f"5. En el escenario 'frases nuevas' la accuracy baja o se mantiene en {acc_nuevas:.3f}, "
      f"lo que indica la capacidad de generalización.")
print(f"6. Mejor intención reconocida: {mejor.Intencion} (F1 = {mejor['Regresión logística (char)']:.2f}); "
      f"peor: {peor.Intencion} (F1 = {peor['Regresión logística (char)']:.2f}).")
print(f"7. Con confianza >= 0.7 el modelo cubre {np.mean(pmax >= 0.7):.1%} de las preguntas; "
      "ese umbral es razonable para decidir cuándo el asistente puede responder solo y cuándo preguntar.")